# ROGII - Kaggle Inference (A4 Multi-Seed)

Internet ON for git clone. Model from `rogii-models-a4-multiseed` dataset. Multi-seed ensemble [42,7,123].

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

INPUT_ROOT = Path('/kaggle/input')
OUTPUT_PATH = Path('/kaggle/working/submission.csv')
REPO_URL = 'https://github.com/Lainterus1/ROGII_Kaggle_Competitions.git'
MODEL_DATASET_SLUG = 'rogii-models-a4-multiseed'
MODEL_FILE = 'baseline_lgbm.pkl'
COMPETITION_SLUG = 'rogii-wellbore-geology-prediction'

# 1. Clone repo from GitHub
repo_root = Path('/kaggle/working/ROGII_Kaggle_Competitions')
if not (repo_root / 'scripts' / 'run_predict.py').is_file():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(repo_root)], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(repo_root / 'requirements.txt'), '--quiet'], cwd=repo_root, check=True)
    print('Repo cloned and dependencies installed.')
else:
    print('Repo already exists.')

# 2. Find model
model_candidates = [p for p in INPUT_ROOT.rglob(MODEL_FILE) if p.is_file()]
if not model_candidates:
    raise FileNotFoundError(f'Could not find {MODEL_FILE}; attach {MODEL_DATASET_SLUG}')
model_path = sorted(model_candidates, key=lambda p: (
    0 if MODEL_DATASET_SLUG.lower() in str(p).replace('\\', '/').lower() else 1,
    len(p.parts),
    str(p),
))[0]

# 3. Find data
data_candidates = []
for current, dirs, files in os.walk(INPUT_ROOT):
    path = Path(current)
    if 'sample_submission.csv' in files and 'test' in dirs:
        data_candidates.append(path)
if not data_candidates:
    raise FileNotFoundError(f'Could not find competition data; attach {COMPETITION_SLUG}')
data_dir = sorted(data_candidates, key=lambda p: (
    0 if COMPETITION_SLUG.lower() in str(p).replace('\\', '/').lower() else 1,
    len(p.parts),
    str(p),
))[0]

print('Repo root:', repo_root)
print('Model path:', model_path)
print('Data dir:', data_dir)
print('Output path:', OUTPUT_PATH)

# 4. Run prediction
cmd = [
    sys.executable,
    str(repo_root / 'scripts' / 'run_predict.py'),
    '--data-dir', str(data_dir),
    '--model', str(model_path),
    '--output', str(OUTPUT_PATH),
    '--savgol-smooth',
]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=repo_root, check=True)

if not OUTPUT_PATH.is_file():
    raise FileNotFoundError(f'Missing expected submission file: {OUTPUT_PATH}')
if OUTPUT_PATH.stat().st_size <= 0:
    raise ValueError(f'Empty submission file: {OUTPUT_PATH}')
print(f'Submission ready: {OUTPUT_PATH} ({OUTPUT_PATH.stat().st_size} bytes)')
